# Stretch experiments: standalone Colab notebook
Upload this file to Colab (**File → Upload notebook**), set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

It needs no GitHub, no Google Drive and no other files, because all of the project code is written out below.
It runs only the stretch experiments:
* **GATv2** on Cora and CiteSeer (10 seeds)
* **GATv2** on the noisy Cora graphs (E3, 5 noise levels × 10 seeds)
* **MLP, GCN, SAGE, GAT and GATv2** on the low-homophily graphs Texas and Chameleon (10 splits)

When it finishes, `results_stretch.zip` downloads automatically. **Keep the tab open until the download finishes.**

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!pip -q install torch_geometric
import os, torch, torch_geometric
os.makedirs("/content/gat/src", exist_ok=True); os.makedirs("/content/gat/scripts", exist_ok=True)
os.chdir("/content/gat")
print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| GPU:", torch.cuda.is_available())

## Project code (written to files)

In [ ]:
%%writefile /content/gat/src/__init__.py
"""Graph Attention Networks vs GCN / GraphSAGE — CS545 project code."""


In [ ]:
%%writefile /content/gat/src/data.py
"""Dataset loading and simple graph statistics."""
import torch
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork

PLANETOID = {"cora": "Cora", "citeseer": "CiteSeer", "pubmed": "PubMed"}
WEBKB = {"texas": "Texas", "cornell": "Cornell", "wisconsin": "Wisconsin"}
WIKI = {"chameleon": "chameleon", "squirrel": "squirrel"}


def load_dataset(name, root="data"):
    """Return (dataset, data) with row-normalised features.

    Planetoid graphs use the standard public split (Cora: 140/500/1000).
    Low-homophily graphs (Texas, Chameleon, ...) ship 10 splits; pick one with
    `select_split(data, i)`.
    """
    key = name.lower()
    transform = T.NormalizeFeatures()
    if key in PLANETOID:
        ds = Planetoid(root, PLANETOID[key], split="public", transform=transform)
    elif key in WEBKB:
        ds = WebKB(root, WEBKB[key], transform=T.Compose([T.ToUndirected(), transform]))
    elif key in WIKI:
        ds = WikipediaNetwork(root, WIKI[key], geom_gcn_preprocess=True,
                              transform=T.Compose([T.ToUndirected(), transform]))
    else:
        raise ValueError(f"Unknown dataset {name}")
    return ds, ds[0]


def select_split(data, split_idx=0):
    """If masks are 2-D (multiple splits), keep only column `split_idx`."""
    data = data.clone()
    for m in ("train_mask", "val_mask", "test_mask"):
        mask = getattr(data, m)
        if mask.dim() == 2:
            setattr(data, m, mask[:, split_idx % mask.size(1)])
    return data


def edge_homophily(edge_index, y):
    """Fraction of edges whose two endpoints share a label (self-loops excluded)."""
    src, dst = edge_index
    keep = src != dst
    return (y[src[keep]] == y[dst[keep]]).float().mean().item()


def describe(name, data):
    """One-row summary used in the notebook's data table."""
    deg = torch.bincount(data.edge_index[1], minlength=data.num_nodes)
    return {
        "dataset": name,
        "nodes": data.num_nodes,
        "edges (undirected)": data.edge_index.size(1) // 2,
        "features": data.num_features,
        "classes": int(data.y.max()) + 1,
        "train/val/test": f"{int(data.train_mask.sum())}/{int(data.val_mask.sum())}/{int(data.test_mask.sum())}",
        "mean degree": round(deg.float().mean().item(), 2),
        "isolated nodes": int((deg == 0).sum()),
        "edge homophily": round(edge_homophily(data.edge_index, data.y), 3),
    }


In [ ]:
%%writefile /content/gat/src/gat_layer.py
"""A Graph Attention layer written from scratch (Velickovic et al., 2018).

For every edge j -> i and every head k:

    e_ij   = LeakyReLU( a_src^k . W^k h_j  +  a_dst^k . W^k h_i )     (= a^T [W h_i || W h_j])
    alpha_ij = softmax over j in N(i) ∪ {i} of e_ij
    h_i'   = sum_j alpha_ij W^k h_j

Heads are concatenated (hidden layers) or averaged (output layer).

Only plain PyTorch ops are used; `check_against_pyg` copies the weights into
PyG's GATConv and confirms both layers produce identical outputs.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def segment_softmax(scores, index, num_nodes):
    """Softmax of `scores` [E, H] within groups of edges sharing the same target `index` [E]."""
    idx = index.unsqueeze(-1).expand_as(scores)
    # subtract per-group max for numerical stability
    group_max = torch.full((num_nodes, scores.size(1)), float("-inf"), device=scores.device, dtype=scores.dtype)
    group_max = group_max.scatter_reduce(0, idx, scores, reduce="amax", include_self=True)
    exp = (scores - group_max[index]).exp()
    denom = torch.zeros((num_nodes, scores.size(1)), device=scores.device, dtype=scores.dtype)
    denom = denom.index_add(0, index, exp)
    return exp / (denom[index] + 1e-16)


def add_self_loops(edge_index, num_nodes):
    """Drop existing self-loops, then append one self-loop per node (same as GATConv)."""
    src, dst = edge_index
    keep = src != dst
    loops = torch.arange(num_nodes, device=edge_index.device)
    return torch.cat([edge_index[:, keep], torch.stack([loops, loops])], dim=1)


class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=1, concat=True, negative_slope=0.2,
                 dropout=0.0, self_loops=True, bias=True):
        super().__init__()
        self.in_dim, self.out_dim, self.heads, self.concat = in_dim, out_dim, heads, concat
        self.negative_slope, self.dropout, self.self_loops = negative_slope, dropout, self_loops

        self.W = nn.Linear(in_dim, heads * out_dim, bias=False)
        self.att_src = nn.Parameter(torch.empty(1, heads, out_dim))  # scores the neighbour j
        self.att_dst = nn.Parameter(torch.empty(1, heads, out_dim))  # scores the centre node i
        self.bias = nn.Parameter(torch.zeros(heads * out_dim if concat else out_dim)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, return_attention_weights=False):
        N, H, C = x.size(0), self.heads, self.out_dim
        if self.self_loops:
            edge_index = add_self_loops(edge_index, N)
        src, dst = edge_index  # message flows src (j) -> dst (i)

        h = self.W(x).view(N, H, C)                          # W h            [N, H, C]
        score_src = (h * self.att_src).sum(-1)               # a_src . W h_j  [N, H]
        score_dst = (h * self.att_dst).sum(-1)               # a_dst . W h_i  [N, H]
        e = F.leaky_relu(score_src[src] + score_dst[dst], self.negative_slope)  # [E, H]

        alpha = segment_softmax(e, dst, N)                   # normalise over each neighbourhood
        alpha_drop = F.dropout(alpha, p=self.dropout, training=self.training)

        out = torch.zeros(N, H, C, device=x.device, dtype=h.dtype)
        out = out.index_add(0, dst, h[src] * alpha_drop.unsqueeze(-1))
        out = out.reshape(N, H * C) if self.concat else out.mean(dim=1)
        if self.bias is not None:
            out = out + self.bias

        if return_attention_weights:
            return out, (edge_index, alpha)
        return out

    def extra_repr(self):
        return f"{self.in_dim}, {self.out_dim}, heads={self.heads}, concat={self.concat}"


@torch.no_grad()
def copy_to_gatconv(layer):
    """Build a PyG GATConv holding exactly the same weights as `layer`."""
    from torch_geometric.nn import GATConv
    conv = GATConv(layer.in_dim, layer.out_dim, heads=layer.heads, concat=layer.concat,
                   negative_slope=layer.negative_slope, dropout=layer.dropout,
                   add_self_loops=layer.self_loops, bias=layer.bias is not None)
    lin = conv.lin if getattr(conv, "lin", None) is not None else conv.lin_src  # PyG >=2.4 vs older
    lin.weight.copy_(layer.W.weight)
    conv.att_src.copy_(layer.att_src)
    conv.att_dst.copy_(layer.att_dst)
    if layer.bias is not None:
        conv.bias.copy_(layer.bias)
    return conv


@torch.no_grad()
def check_against_pyg(x, edge_index, in_dim, out_dim, heads=8, concat=True, seed=0, atol=1e-5):
    """Return max |difference| between our layer and GATConv (outputs and attention)."""
    torch.manual_seed(seed)
    ours = GATLayer(in_dim, out_dim, heads=heads, concat=concat).eval()
    pyg = copy_to_gatconv(ours).eval()
    out_a, (ei_a, att_a) = ours(x, edge_index, return_attention_weights=True)
    out_b, (ei_b, att_b) = pyg(x, edge_index, return_attention_weights=True)

    # GATConv may order edges differently; compare attention per (src, dst) pair
    N = x.size(0)
    key_a, key_b = ei_a[0] * N + ei_a[1], ei_b[0] * N + ei_b[1]
    att_a = att_a[key_a.argsort()]
    att_b = att_b[key_b.argsort()]
    out_diff = (out_a - out_b).abs().max().item()
    att_diff = (att_a - att_b).abs().max().item()
    return {"output_max_abs_diff": out_diff, "attention_max_abs_diff": att_diff,
            "match": out_diff < atol and att_diff < atol}


In [ ]:
%%writefile /content/gat/src/models.py
"""Two-layer models compared in the project, each with its original paper's hyperparameters."""
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATv2Conv

from .gat_layer import GATLayer

# Hyperparameters follow the original papers (GCN: Kipf & Welling 2017; GAT: Velickovic et al. 2018).
CONFIGS = {
    "MLP":   dict(hidden=64, dropout=0.5, lr=0.01,  weight_decay=5e-4),
    "GCN":   dict(hidden=16, dropout=0.5, lr=0.01,  weight_decay=5e-4),
    "SAGE":  dict(hidden=64, dropout=0.5, lr=0.01,  weight_decay=5e-4),
    "GAT":   dict(hidden=8, heads=8, dropout=0.6, lr=0.005, weight_decay=5e-4),
    "GATv2": dict(hidden=8, heads=8, dropout=0.6, lr=0.005, weight_decay=5e-4),
}


class MLP(nn.Module):
    """Ignores the graph entirely: a lower bound for what edges add."""
    uses_graph = False

    def __init__(self, in_dim, n_classes, hidden, dropout, **_):
        super().__init__()
        self.lin1, self.lin2, self.dropout = nn.Linear(in_dim, hidden), nn.Linear(hidden, n_classes), dropout

    def embed(self, x, edge_index=None):
        x = F.dropout(x, self.dropout, self.training)
        return F.relu(self.lin1(x))

    def forward(self, x, edge_index=None):
        return self.lin2(F.dropout(self.embed(x), self.dropout, self.training))


class GCN(nn.Module):
    """Neighbours weighted by 1/sqrt(deg_i deg_j) — fixed by graph structure."""
    uses_graph = True

    def __init__(self, in_dim, n_classes, hidden, dropout, **_):
        super().__init__()
        self.conv1, self.conv2, self.dropout = GCNConv(in_dim, hidden), GCNConv(hidden, n_classes), dropout

    def embed(self, x, edge_index):
        x = F.dropout(x, self.dropout, self.training)
        return F.relu(self.conv1(x, edge_index))

    def forward(self, x, edge_index):
        return self.conv2(F.dropout(self.embed(x, edge_index), self.dropout, self.training), edge_index)


class SAGE(nn.Module):
    """GraphSAGE with mean aggregator: W_self h_i + W_neigh mean_j h_j (full neighbourhood)."""
    uses_graph = True

    def __init__(self, in_dim, n_classes, hidden, dropout, **_):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden, aggr="mean")
        self.conv2 = SAGEConv(hidden, n_classes, aggr="mean")
        self.dropout = dropout

    def embed(self, x, edge_index):
        x = F.dropout(x, self.dropout, self.training)
        return F.relu(self.conv1(x, edge_index))

    def forward(self, x, edge_index):
        return self.conv2(F.dropout(self.embed(x, edge_index), self.dropout, self.training), edge_index)


class GAT(nn.Module):
    """GAT as in the paper: 8 heads x 8 units + ELU, then one output head. `v2=True` swaps in GATv2."""
    uses_graph = True

    def __init__(self, in_dim, n_classes, hidden, heads, dropout, v2=False, **_):
        super().__init__()
        if v2:
            self.conv1 = GATv2Conv(in_dim, hidden, heads=heads, dropout=dropout)
            self.conv2 = GATv2Conv(hidden * heads, n_classes, heads=1, concat=False, dropout=dropout)
        else:
            self.conv1 = GATLayer(in_dim, hidden, heads=heads, dropout=dropout)
            self.conv2 = GATLayer(hidden * heads, n_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def embed(self, x, edge_index):
        x = F.dropout(x, self.dropout, self.training)
        return F.elu(self.conv1(x, edge_index))

    def forward(self, x, edge_index, return_attention=False):
        x = F.dropout(x, self.dropout, self.training)
        h, att1 = self.conv1(x, edge_index, return_attention_weights=True)
        h = F.dropout(F.elu(h), self.dropout, self.training)
        out, att2 = self.conv2(h, edge_index, return_attention_weights=True)
        if return_attention:
            return out, [att1, att2]  # each: (edge_index incl. self-loops, alpha [E, heads])
        return out


def build_model(name, in_dim, n_classes):
    cfg = CONFIGS[name]
    if name == "MLP":
        return MLP(in_dim, n_classes, **cfg)
    if name == "GCN":
        return GCN(in_dim, n_classes, **cfg)
    if name == "SAGE":
        return SAGE(in_dim, n_classes, **cfg)
    if name == "GAT":
        return GAT(in_dim, n_classes, **cfg)
    if name == "GATv2":
        return GAT(in_dim, n_classes, v2=True, **cfg)
    raise ValueError(name)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile /content/gat/src/train.py
"""Full-batch training with early stopping on validation accuracy."""
import copy
import random
import time

import numpy as np
import torch
import torch.nn.functional as F

from .models import CONFIGS, build_model, count_params


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(-1)
    res = {}
    for split in ("train", "val", "test"):
        mask = getattr(data, f"{split}_mask")
        res[f"{split}_acc"] = (pred[mask] == data.y[mask]).float().mean().item()
        res[f"{split}_loss"] = F.cross_entropy(logits[mask], data.y[mask]).item()
    return res


def train_model(name, data, seed=0, max_epochs=1000, patience=100, device=None, verbose=False):
    """Train one model; keep the weights with best val accuracy (ties -> lower val loss).

    Returns (model with best weights loaded, metrics dict).
    """
    device = device or get_device()
    data = data.to(device)
    set_seed(seed)
    n_classes = int(data.y.max()) + 1
    model = build_model(name, data.num_features, n_classes).to(device)
    cfg = CONFIGS[name]
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best = (-1.0, float("inf"))
    best_state, best_epoch, bad, epoch_times = None, 0, 0, []
    for epoch in range(1, max_epochs + 1):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - t0)

        m = evaluate(model, data)
        score = (m["val_acc"], m["val_loss"])
        if score[0] > best[0] or (score[0] == best[0] and score[1] < best[1]):
            best, best_epoch, bad = score, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
        if verbose and epoch % 50 == 0:
            print(f"{name} ep {epoch:4d} loss {loss.item():.4f} val {m['val_acc']:.3f}")
        if bad >= patience:
            break

    model.load_state_dict(best_state)
    final = evaluate(model, data)
    metrics = {
        "model": name, "seed": seed,
        "test_acc": final["test_acc"], "val_acc": final["val_acc"], "train_acc": final["train_acc"],
        "best_epoch": best_epoch, "epochs_run": epoch,
        "ms_per_epoch": 1000 * float(np.mean(epoch_times[1:] or epoch_times)),  # skip warm-up epoch
        "params": count_params(model),
    }
    return model, metrics


In [ ]:
%%writefile /content/gat/src/noise.py
"""Inject random (noisy) undirected edges into a graph — experiment E3."""
import numpy as np
import torch


def add_random_edges(edge_index, num_nodes, ratio, seed=0):
    """Add `ratio * (#undirected edges)` new random undirected edges.

    New edges join uniformly random node pairs that are not already connected
    (no self-loops, no duplicates). With 7 roughly balanced classes most of them
    are cross-class by chance.

    Returns (new_edge_index, is_noise) where is_noise[k] is True for injected edges.
    Original edges come first, so `new_edge_index[:, :E]` equals the input.
    """
    E = edge_index.size(1)
    n_new = int(round(ratio * (E // 2)))
    if n_new == 0:
        return edge_index.clone(), torch.zeros(E, dtype=torch.bool)

    rng = np.random.default_rng(seed)
    ei = edge_index.cpu().numpy()
    existing = set((ei[0] * num_nodes + ei[1]).tolist())
    new = set()
    while len(new) < n_new:
        k = (n_new - len(new)) * 2
        a = rng.integers(0, num_nodes, k)
        b = rng.integers(0, num_nodes, k)
        for i, j in zip(a.tolist(), b.tolist()):
            if i == j:
                continue
            i, j = min(i, j), max(i, j)
            key = i * num_nodes + j
            if key in existing or key in new:
                continue
            new.add(key)
            if len(new) == n_new:
                break

    new = np.array(sorted(new))
    i, j = new // num_nodes, new % num_nodes
    added = torch.tensor(np.stack([np.concatenate([i, j]), np.concatenate([j, i])]),
                         dtype=edge_index.dtype, device=edge_index.device)
    new_ei = torch.cat([edge_index, added], dim=1)
    is_noise = torch.cat([torch.zeros(E, dtype=torch.bool), torch.ones(added.size(1), dtype=torch.bool)])
    return new_ei, is_noise.to(edge_index.device)


def with_noise(data, ratio, seed=0):
    """Copy of `data` with injected edges; also stores `data.is_noise` for each edge."""
    d = data.clone()
    d.edge_index, d.is_noise = add_random_edges(data.edge_index, data.num_nodes, ratio, seed)
    return d


In [ ]:
%%writefile /content/gat/src/attention.py
"""Tools for reading what a trained GAT pays attention to (E2, E3)."""
import numpy as np
import pandas as pd
import torch


@torch.no_grad()
def get_attention(model, data):
    """Return per-layer list of (edge_index incl. self-loops, alpha averaged over heads [E])."""
    model.eval()
    _, atts = model(data.x, data.edge_index, return_attention=True)
    return [(ei.cpu(), a.mean(dim=1).cpu()) for ei, a in atts]


def node_attention_stats(edge_index, alpha, y, num_nodes):
    """Per-node statistics of one layer's attention.

    entropy          : -sum_j alpha_ij log alpha_ij over N(i) ∪ {i}
    uniform_entropy  : log(deg_i + 1), entropy if every neighbour got the same weight
    self_att         : weight on the node itself
    same_class_att   : share of *neighbour* attention (self excluded, renormalised) on same-class neighbours
    same_class_frac  : share of neighbours with the same class = what uniform weights would give
    """
    y = y.cpu()
    src, dst = edge_index
    is_self = src == dst
    ent_terms = -(alpha * torch.log(alpha.clamp_min(1e-12)))
    entropy = torch.zeros(num_nodes).index_add(0, dst, ent_terms)
    deg = torch.bincount(dst[~is_self], minlength=num_nodes).float()
    self_att = torch.zeros(num_nodes).index_add(0, dst[is_self], alpha[is_self])

    ns, nd, na = src[~is_self], dst[~is_self], alpha[~is_self]
    same = (y[ns] == y[nd]).float()
    neigh_att = torch.zeros(num_nodes).index_add(0, nd, na)
    same_att = torch.zeros(num_nodes).index_add(0, nd, na * same)
    same_cnt = torch.zeros(num_nodes).index_add(0, nd, same)

    df = pd.DataFrame({
        "degree": deg.numpy(),
        "entropy": entropy.numpy(),
        "uniform_entropy": torch.log(deg + 1).numpy(),
        "self_att": self_att.numpy(),
        "same_class_att": (same_att / neigh_att.clamp_min(1e-12)).numpy(),
        "same_class_frac": (same_cnt / deg.clamp_min(1)).numpy(),
        "label": y.numpy(),
    })
    df.loc[df.degree == 0, ["same_class_att", "same_class_frac"]] = np.nan
    return df


def normalized_edge_attention(edge_index, alpha, num_nodes):
    """alpha_ij * (deg_i + 1): 1.0 means 'exactly uniform', >1 above average, <1 below."""
    dst = edge_index[1]
    size = torch.bincount(dst, minlength=num_nodes).float()  # deg + self-loop
    return alpha * size[dst]


def edge_mask_for(edge_index, ref_edge_index, ref_mask, num_nodes):
    """Map a boolean mask defined on `ref_edge_index` onto the (reordered) `edge_index`."""
    keys = edge_index[0].long() * num_nodes + edge_index[1].long()
    ref_keys = (ref_edge_index[0].long() * num_nodes + ref_edge_index[1].long()).cpu()
    return torch.isin(keys, ref_keys[ref_mask.cpu()])


def noise_attention_summary(model, data):
    """Mean normalised attention on original vs injected edges, per layer (self-loops excluded)."""
    rows = []
    for layer, (ei, a) in enumerate(get_attention(model, data), start=1):
        norm = normalized_edge_attention(ei, a, data.num_nodes)
        noise = edge_mask_for(ei, data.edge_index, data.is_noise, data.num_nodes)
        not_self = ei[0] != ei[1]
        rows.append({
            "layer": layer,
            "att_original": norm[not_self & ~noise].mean().item(),
            "att_injected": norm[not_self & noise].mean().item() if (noise.any()) else float("nan"),
        })
    return rows


In [ ]:
%%writefile /content/gat/scripts/run_experiments.py
"""Run the project's experiments and append results to CSV files in results/.

    python scripts/run_experiments.py --exp e1                # benchmark, Cora + CiteSeer, 10 seeds
    python scripts/run_experiments.py --exp e3                # noisy edges on Cora
    python scripts/run_experiments.py --exp e1 --models GAT GATv2 --datasets Texas Chameleon   # stretch

Runs already present in the CSV are skipped, so an interrupted Colab session can simply be re-run.
"""
import argparse
import os
import sys
import time

import pandas as pd
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from src.attention import noise_attention_summary  # noqa: E402
from src.data import load_dataset, select_split  # noqa: E402
from src.noise import with_noise  # noqa: E402
from src.train import get_device, train_model  # noqa: E402

ALL_MODELS = ["MLP", "GCN", "SAGE", "GAT"]


def done_keys(path, cols):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    return set(map(tuple, df[cols].astype(str).values.tolist()))


def append_row(path, row):
    """Rewrite the whole CSV so rows with extra columns (GAT attention stats) keep a valid header."""
    df = pd.DataFrame([row])
    if os.path.exists(path):
        df = pd.concat([pd.read_csv(path), df], ignore_index=True)
    df.to_csv(path, index=False)


def run_e1(args, device):
    path = os.path.join(args.out, "e1_benchmark.csv")
    done = done_keys(path, ["dataset", "model", "seed"])
    for ds_name in args.datasets or ["Cora", "CiteSeer"]:
        _, data = load_dataset(ds_name, args.data_root)
        for seed in range(args.seeds):
            d = select_split(data, seed)  # no-op for Planetoid's single public split
            for name in args.models or ALL_MODELS:
                if (ds_name, name, str(seed)) in done:
                    continue
                model, m = train_model(name, d, seed=seed, device=device)
                append_row(path, {"dataset": ds_name, **m})
                print(f"[E1] {ds_name:9s} {name:6s} seed {seed}: test {m['test_acc']:.3f} "
                      f"(best ep {m['best_epoch']}, {m['ms_per_epoch']:.1f} ms/ep)", flush=True)
                if name in ("GAT", "GATv2") and seed == 0:  # keep one trained GAT for E2 plots
                    torch.save(model.state_dict(), os.path.join(args.out, f"{name}_{ds_name}_seed0.pt"))


def run_e3(args, device):
    path = os.path.join(args.out, "e3_noise.csv")
    done = done_keys(path, ["dataset", "noise", "model", "seed"])
    for ds_name in args.datasets or ["Cora"]:
        _, data = load_dataset(ds_name, args.data_root)
        for ratio in args.noise:
            for seed in range(args.seeds):
                noisy = with_noise(select_split(data, seed), ratio, seed=seed)  # same noisy graph for every model
                y = noisy.y
                src, dst = noisy.edge_index[:, noisy.is_noise]
                cross = (y[src] != y[dst]).float().mean().item() if src.numel() else float("nan")
                for name in args.models or ALL_MODELS:
                    if (ds_name, str(ratio), name, str(seed)) in done:
                        continue
                    model, m = train_model(name, noisy, seed=seed, device=device)
                    row = {"dataset": ds_name, "noise": ratio, "injected_cross_class": cross, **m}
                    if name in ("GAT", "GATv2"):
                        for r in noise_attention_summary(model, noisy.to(device)):
                            row[f"L{r['layer']}_att_original"] = r["att_original"]
                            row[f"L{r['layer']}_att_injected"] = r["att_injected"]
                    append_row(path, row)
                    print(f"[E3] {ds_name} noise {ratio:.2f} {name:6s} seed {seed}: test {m['test_acc']:.3f}",
                          flush=True)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--exp", choices=["e1", "e3"], required=True)
    p.add_argument("--datasets", nargs="+")
    p.add_argument("--models", nargs="+")
    p.add_argument("--seeds", type=int, default=10)
    p.add_argument("--noise", type=float, nargs="+", default=[0.0, 0.1, 0.25, 0.5, 1.0])
    p.add_argument("--out", default="results")
    p.add_argument("--data_root", default="data")
    args = p.parse_args()
    os.makedirs(args.out, exist_ok=True)
    device = get_device()
    print("device:", device, "| torch", torch.__version__)
    t0 = time.time()
    {"e1": run_e1, "e3": run_e3}[args.exp](args, device)
    print(f"done in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    main()


## Run the stretch experiments
The first command takes longest (Chameleon). Progress is printed per run.

In [ ]:
import os; os.chdir("/content/gat")
!python -W ignore scripts/run_experiments.py --exp e1 --models MLP GCN SAGE GAT GATv2 --datasets Texas Chameleon --out results_stretch

In [ ]:
import os; os.chdir("/content/gat")
!python -W ignore scripts/run_experiments.py --exp e1 --models GATv2 --datasets Cora CiteSeer --out results_stretch

In [ ]:
import os; os.chdir("/content/gat")
!python -W ignore scripts/run_experiments.py --exp e3 --models GATv2 --out results_stretch

## Quick look and download

In [ ]:
import os, pandas as pd
os.chdir("/content/gat")
e1 = pd.read_csv("results_stretch/e1_benchmark.csv")
display((e1.groupby(["dataset", "model"]).test_acc.agg(["mean", "std", "count"]) * [100, 100, 1]).round(2))
e3 = pd.read_csv("results_stretch/e3_noise.csv")
display((e3.groupby("noise").test_acc.agg(["mean", "std"]) * 100).round(2))

In [ ]:
import os; os.chdir("/content/gat")
!zip -qr /content/results_stretch.zip results_stretch
from google.colab import files
files.download("/content/results_stretch.zip")